In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. مكتبات تقسيم البيانات والتقييم
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ---------------------------------------------------------
# 2. استيراد الـ 20 خوارزمية (جميعها من scikit-learn الرسمية)
# ---------------------------------------------------------
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Perceptron, SGDClassifier, PassiveAggressiveClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# ---------------------------------------------------------
# 3. إعداد مسارات الملفات للتحويلات الأربعة
# ---------------------------------------------------------
datasets = {
    "MFL (Most Frequent Label)": "yeast_single_label_mfl.csv",
    "LFL (Least Frequent Label)": "yeast_single_label_lfl.csv",
    "LP (Label Powerset)": "yeast_single_label_lp.csv",
    "IGNORE (Filtered Multi-Label)": "yeast_single_label_ignore.csv"
}

# ---------------------------------------------------------
# 4. تعريف قائمة الـ 20 خوارزمية (محسّنة للسرعة والتوازي)
# ---------------------------------------------------------
models = {
    # Linear Models
    "1. Logistic Regression": LogisticRegression(max_iter=100, n_jobs=-1, random_state=42),
    "2. Ridge Classifier": RidgeClassifier(),
    "3. Perceptron": Perceptron(max_iter=200, n_jobs=-1, random_state=42),
    "4. SGD Classifier": SGDClassifier(max_iter=500, n_jobs=-1, random_state=42),
    "5. Passive Aggressive Classifier": PassiveAggressiveClassifier(max_iter=300, n_jobs=-1, random_state=42),

    # Naive Bayes
    "6. Gaussian Naive Bayes": GaussianNB(),
    "7. Bernoulli Naive Bayes": BernoulliNB(),

    # K-Nearest Neighbors & Discriminant Analysis
    "8. K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "9. Linear Discriminant Analysis (LDA)": LinearDiscriminantAnalysis(),

    # Support Vector Machines (وضع max_iter لمنع التعليق)
    "10. Support Vector Classifier (SVM RBF)": SVC(kernel='rbf', max_iter=1000, random_state=42),
    "11. Linear SVC": SVC(kernel='linear', max_iter=500, random_state=42),

    # Decision Trees
    "12. Decision Tree": DecisionTreeClassifier(random_state=42),
    "13. Extra Tree Classifier": ExtraTreeClassifier(random_state=42),

    # Bagging & Forest Ensembles (إضافة max_depth و n_jobs=-1)
    "14. Random Forest": RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42),
    "15. Extra Trees Ensemble": ExtraTreesClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42),
    "16. Bagging Classifier": BaggingClassifier(n_estimators=20, n_jobs=-1, random_state=42),

    # Boosting Ensembles (ضبط n_estimators وتخفيف العمق)
    "17. Gradient Boosting": GradientBoostingClassifier(n_estimators=30, max_depth=3, random_state=42),
    "18. AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
    "19. Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=50, random_state=42),

    # Neural Networks (تحديد التكرارات والتوقف المبكر)
    "20. Multi-layer Perceptron (MLP/ANN)": MLPClassifier(hidden_layer_sizes=(30,), max_iter=50, early_stopping=True, random_state=42)
}

# هياكل بيانات لتخزين وإحصاء المقارنات النهائية
all_experiments = []
best_model_per_transform = {}
transform_overall_scores = {}

# ---------------------------------------------------------
# 5. التكرار والتدريب على كل تحويل
# ---------------------------------------------------------
for transformation_name, file_path in datasets.items():
    print("\n" + "#"*100)
    print(f"   النتائج الخاصة بالتحويل: {transformation_name}")
    print(f"   الملف: {file_path}")
    print("#"*100)

    # قراءة البيانات
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    # فحص خيار التقسيم تلقائياً لتفادي ValueError
    use_stratify = (y.value_counts().min() >= 2)

    # تقسيم البيانات
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y if use_stratify else None
    )

    print(f"--> عدد عينات التدريب: {X_train.shape[0]} | عدد عينات الاختبار: {X_test.shape[0]}\n")

    results = []

    # تدريب الخوارزميات
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec
        })

        all_experiments.append({
            "Transformation": transformation_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec
        })

    # تحويل نتائج التحويلة لـ DataFrame وترتيبها
    results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

    # عرض الجدول كاملاً
    print("="*95)
    print(f"{'الخوارزمية (Algorithm)':<40} | {'Accuracy':<12} | {'Precision':<12} | {'Recall':<12}")
    print("="*95)
    for idx, row in results_df.iterrows():
        print(f"{row['Model']:<40} | {row['Accuracy']*100:6.2f}%      | {row['Precision']:6.4f}       | {row['Recall']:6.4f}")

    best_single_model = results_df.iloc[0]
    best_model_per_transform[transformation_name] = best_single_model
    transform_overall_scores[transformation_name] = results_df["Accuracy"].mean()

# ---------------------------------------------------------
# 6. التقرير النهائي والمقارنات الشاملة
# ---------------------------------------------------------
all_experiments_df = pd.DataFrame(all_experiments)

print("\n" + "="*100)
print("                                  الخلاصة والنتائج النهائية")
print("="*100)

# 1. أفضل خوارزمية لكل تحويلة
print("\n[1] أفضل خوارزمية لكل تحويلة على حدة (Best Model Per Transformation):")
print("-"*90)
for trans, best in best_model_per_transform.items():
    print(f"• التحويلة: {trans:<30} -> أفضل خوارزمية: {best['Model']:<35} (Accuracy: {best['Accuracy']*100:.2f}%)")

# 2. أفضل تحويل من الأربعة
best_overall_transformation = max(transform_overall_scores, key=transform_overall_scores.get)
print("\n" + "-"*90)
print(f"[2] أفضل تحويل من التحويلات الأربعة بشكل عام (Highest Average Accuracy):")
print(f"--> {best_overall_transformation} (بمتوسط دقة قدره: {transform_overall_scores[best_overall_transformation]*100:.2f}%)")

# 3. أفضل تركيبة مطلقة
top_combination = all_experiments_df.sort_values(by="Accuracy", ascending=False).iloc[0]
print("\n" + "-"*90)
print(f"[3] أفضل تركيبة متميزة شاملة (Best Transformation + Algorithm Combination):")
print(f"--> التحويلة:    {top_combination['Transformation']}")
print(f"--> الخوارزمية:  {top_combination['Model']}")
print(f"--> نسبة الدقة:  {top_combination['Accuracy']*100:.2f}%")
print(f"--> Precision:   {top_combination['Precision']:.4f}")
print(f"--> Recall:      {top_combination['Recall']:.4f}")
print("="*100)


####################################################################################################
   النتائج الخاصة بالتحويل: MFL (Most Frequent Label)
   الملف: yeast_single_label_mfl.csv
####################################################################################################
--> عدد عينات التدريب: 1933 | عدد عينات الاختبار: 484

الخوارزمية (Algorithm)                   | Accuracy     | Precision    | Recall      
15. Extra Trees Ensemble                 |  37.19%      | 0.2458       | 0.3719
10. Support Vector Classifier (SVM RBF)  |  36.78%      | 0.2086       | 0.3678
14. Random Forest                        |  36.78%      | 0.1677       | 0.3678
18. AdaBoost                             |  35.54%      | 0.1991       | 0.3554
19. Hist Gradient Boosting               |  33.68%      | 0.2045       | 0.3368
2. Ridge Classifier                      |  33.26%      | 0.2165       | 0.3326
20. Multi-layer Perceptron (MLP/ANN)     |  33.06%      | 0.2322       | 0.3306
17. G